# Avatar Chatbot v0.2 - Mit LangGraph Memory 🧠

Ein CV-basierter Chatbot mit **Konversations-Memory** und **Multi-Thread Support**.

**Unterschiede zu v0.1:**
- ✅ LangGraph für State Management
- ✅ Follow-up Fragen möglich ("Und danach? Wo hat er arbeitet?")
- ✅ Multi-User Support (separate Threads)
- ✅ Automatische Memory-Verwaltung
- ✅ Pydantic für strukturierte Ausgabe

## 1️⃣ Setup & Environment

In [ ]:
import subprocess
import sys

# Installiere fehlende Pakete
try:
    from dotenv import load_dotenv
    from langgraph.graph import StateGraph, MessagesState, START, END
    from langgraph.checkpoint.memory import MemorySaver
    from pydantic import BaseModel, Field
except ImportError:
    print("📦 Installiere fehlende Pakete...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "langgraph", "python-dotenv", "pydantic"])
    from dotenv import load_dotenv
    from langgraph.graph import StateGraph, MessagesState, START, END
    from langgraph.checkpoint.memory import MemorySaver
    from pydantic import BaseModel, Field

import os
from pathlib import Path

# 🔧 FIX: Stelle sicher, dass wir im richtigen Docker-Verzeichnis sind
os.chdir('/workspace')
print(f"📂 Working directory: {os.getcwd()}")

# API Keys laden (relative path works in Docker and locally)
env_path = Path('./.env')
if env_path.exists():
    load_dotenv(env_path)
else:
    # Try to load from environment directly if .env doesn't exist
    load_dotenv()

# Imports
from genai_lib.utilities import check_environment, mprint
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

print("✅ Umgebung vorbereitet")
check_environment()
print()
print(f"✓ OPENAI_API_KEY gesetzt: {'OPENAI_API_KEY' in os.environ and os.environ['OPENAI_API_KEY'] != ''}")

## 2️⃣ CV laden

In [ ]:
# CV.md laden (relative path für Docker compatibility)
cv_path = Path('./tasks/Avatar/CV.md')

# Debug: Zeige aktuelles Verzeichnis
print(f"📂 Current directory: {Path.cwd()}")
print(f"📂 Looking for: {cv_path}")
print(f"📂 Absolute path: {cv_path.absolute()}")
print(f"📂 File exists: {cv_path.exists()}")

if not cv_path.exists():
    print(f"❌ Error: CV not found at {cv_path.absolute()}")
    print(f"📁 Available files in ./tasks/Avatar/:")
    import subprocess
    subprocess.run(['ls', '-la', './tasks/Avatar/'], capture_output=False)
    raise FileNotFoundError(f"CV.md not found at {cv_path}")

with open(cv_path, 'r', encoding='utf-8') as f:
    cv_content = f.read()

# Preview
mprint("### 📄 CV Preview")
mprint("---")
mprint(cv_content[:500] + "...")

## 3️⃣ LangGraph Setup

In [ ]:
# System-Prompt (wie in v0.1, aber jetzt für LangGraph)
system_prompt = f"""Du bist ein hilfreicher Assistent, der Fragen über eine Person beantwortet.

Hier ist der Lebenslauf der Person:

{cv_content}

ICHTIGE REGELN:
1. Beantworte Fragen NUR auf Basis der bereitgestellten Lebenslauf-Informationen
2. Wenn eine Information NICHT im Lebenslauf vorhanden ist, antworte GENAU mit:
   "Das weiss ich leider nicht"
3. Erfinde KEINE Informationen - sei ehrlich, wenn du etwas nicht weisst
4. Antworte auf Deutsch
5. Sei präzise und konkret in deinen Antworten"""

print("✅ System-Prompt definiert")

## 4️⃣ LLM & Modell

In [ ]:
# Modell initialisieren
model_name = "gpt-4o-mini"
temperature = 0.0
llm = init_chat_model(model_name, model_provider="openai", temperature=temperature)

print(f"✅ Modell: {model_name}")
print(f"✅ Temperature: {temperature}")

## 5️⃣ Chat-Node mit MessagesState

In [ ]:
# Chat-Node: Wird bei jedem Aufruf aufgerufen
def chat_node(state: MessagesState):
    """
    Diese Funktion wird bei jedem Chat-Schritt aufgerufen.
    Der 'state' enthält automatisch alle bisherigen Nachrichten.
    """
    # System-Prompt + vollständige Historie
    messages = [
        SystemMessage(content=system_prompt)
    ] + state["messages"]

    # LLM aufrufen
    response = llm.invoke(messages)

    # WICHTIG: Die Rückgabe MUSS eine Liste von Nachrichten sein!
    # Der MessagesState Reducer fügt diese der Historie hinzu.
    return {"messages": [response]}

print("✅ Chat-Node definiert")

## 6️⃣ Graph kompilieren

In [ ]:
# Graph aufbauen
workflow = StateGraph(state_schema=MessagesState)

# Chat-Node hinzufügen
workflow.add_node("chat", chat_node)

# Kanten: START → chat → END
workflow.add_edge(START, "chat")
workflow.add_edge("chat", END)

# Checkpointer: Speichert Memory pro thread_id
checkpointer = MemorySaver()

# Graph kompilieren
graph = workflow.compile(checkpointer=checkpointer)

print("✅ LangGraph kompiliert mit MemorySaver")
print(f"✅ StateGraph: START → chat → END")

## 7️⃣ Chat-Funktion für Multi-Thread

In [ ]:
def chat(thread_id: str, user_input: str) -> str:
    """
    Chattet mit dem Avatar unter einer bestimmten thread_id.
    Jede thread_id hat ihre eigene, isolierte Memory.
    """
    # Config mit thread_id (Session-ID)
    config = {"configurable": {"thread_id": thread_id}}

    # Input vorbereiten (neue User-Nachricht)
    input_state = {"messages": [HumanMessage(content=user_input)]}

    # Graph aufrufen - Memory wird vom Checkpointer automatisch geladen/gespeichert!
    result = graph.invoke(input_state, config=config)

    # Letzte AI-Nachricht extrahieren
    response = result["messages"][-1].content

    # Ausgabe
    mprint(f"**👤 [{thread_id}] User:** \n{user_input}")
    mprint(f"**🤖 [{thread_id}] Avatar:** \n{response}\n")

    return response

def show_thread_history(thread_id: str):
    """
    Zeigt die komplette Konversations-Historie eines Threads.
    """
    config = {"configurable": {"thread_id": thread_id}}
    state = graph.get_state(config)
    messages = state.values.get("messages", [])

    mprint(f"### 📝 Thread '{thread_id}' - {len(messages)} Nachrichten")
    mprint("---")

    for i, msg in enumerate(messages, 1):
        role = "👤" if msg.type == "human" else "🤖"
        mprint(f"{i}. {role} [{msg.type.upper()}]: {msg.content}\n")

print("✅ Chat-Funktionen definiert")

## 8️⃣ Test 1: Single User mit Follow-Ups

In [ ]:
mprint("# Test 1: Single User mit Follow-up Fragen 🧑")
mprint("="*60)

thread_max = "user_max_session"

# Erste Frage
chat(thread_max, "Wo hat Max studiert?")

# Follow-up (v0.1 konnte das nicht!)
chat(thread_max, "Und wo arbeitet er jetzt?")

# Noch eine Follow-up
chat(thread_max, "Was sind seine Hobbies?")

# Info die nicht im CV ist
chat(thread_max, "Ist er verheiratet?")

# History anzeigen
mprint("")
show_thread_history(thread_max)

## 9️⃣ Test 2: Multi-User (Parallele Sessions)

In [ ]:
mprint("# Test 2: Multi-User Support 👥👥")
mprint("="*60)

thread_emma = "user_emma_session"

# Emma's erste Frage
chat(thread_emma, "Welche Programmiersprachen beherrscht die Person?")

# Zurück zu Max - Memory ist erhalten!
chat(thread_max, "Was sind die Zertifikate?")

# Zurück zu Emma - ihre Memory ist auch erhalten!
chat(thread_emma, "Hat sie Berufserfahrung mit Python?")

# Zurück zu Max - nochmal testen
chat(thread_max, "Wie viele Jahre Erfahrung insgesamt?")

mprint("")
mprint("## Thread Comparison")
mprint("="*60)
show_thread_history(thread_max)
show_thread_history(thread_emma)

## 🔟 Automatisierte Test-Suite

In [ ]:
# Test-Suite: Mit Memory sollten Follow-ups jetzt funktionieren
test_thread = "test_suite"

test_cases = [
    ("Wo hat Max studiert?", "TU München oder LMU München"),
    ("Und wo arbeitet er jetzt?", "StartupX"),  # Follow-up!
    ("Welche Programmiersprachen?", "Python, SQL, JavaScript"),
    ("Hat er KI-Erfahrung?", "KI" or "Machine Learning" or "Transformer"),
    ("Was sind seine Hobbies?", "Wandern, Open Source, Schach"),
]

mprint("# Automatisierte Test-Suite")
mprint("="*60)
passed = 0
failed = 0

for i, (question, expected) in enumerate(test_cases, 1):
    response = chat(test_thread, question)
    
    # Prüfe, ob expected in der response ist
    if isinstance(expected, str) and expected.lower() in response.lower():
        status = "✅ PASS"
        passed += 1
    elif isinstance(expected, tuple) and any(e.lower() in response.lower() for e in expected.split(" or ")):
        status = "✅ PASS"
        passed += 1
    else:
        status = "❌ FAIL"
        failed += 1
    
    mprint(f"{i}. {status} - {question}")

mprint("")
mprint(f"## 📊 Ergebnisse: {passed} ✅ / {failed} ❌")

## 🎯 Erkenntnisse & Unterschiede zu v0.1

### Was ist neu in v0.2?

#### ✅ LangGraph State Management
- **v0.1:** Manuelles Prompt-Building, jede Frage isoliert
- **v0.2:** Automatische State-Verwaltung mit `MessagesState`

```python
# v0.1
response = chain.invoke({"user_input": "Wo studiert Max?"})

# v0.2
response = graph.invoke({"messages": [HumanMessage("...")]}, config)
# Memory automatisch gespeichert!
```

#### ✅ Follow-up Fragen
- **v0.1:** "Wo arbeitet er jetzt?" → Bot antwortet ohne Kontext
- **v0.2:** "Und wo arbeitet er jetzt?" → Bot versteht "er" = Max

#### ✅ Multi-Thread Support
```python
# Jede thread_id hat separate Memory
chat("user_max", "Ich heiße Max")
chat("user_emma", "Ich heiße Emma")
chat("user_max", "Wie heiße ich?")  # Antwortet "Max"
```

#### ✅ Memory Persistence
- Checkpointer speichert automatisch
- Kein manuelles Listen-Management
- Einfach später zu SQLite erweitbar

### Technische Architektur

```
v0.1 (Simple Chain):
Prompt → LLM → Parser → Output
  ❌ Stateless

v0.2 (LangGraph):
  State: {messages: [...]}
    ↓
  START
    ↓
  chat_node(state) → LLM → Response
    ↓
  State.messages.append(response)
    ↓
  Checkpointer speichert State
    ↓
  END
```

### Limitierungen (noch immer)
- ❌ Context-Größe: Funktioniert für kurze CVs
- ⚠️ Lange Sessions: Memory wächst (später Trimming/Summarization)
- ⚠️ Modell-abhängig: Halluzinationen immer möglich

### Roadmap
- **v0.2** ✅ Memory + Follow-ups
- **v0.3** Pydantic für strukturierte Ausgabe
- **v0.4** Memory Trimming/Summarization
- **v0.5** RAG mit ChromaDB (längere CVs)

## 📚 Best Practices für v0.2

### 1. Thread-IDs verwenden
```python
# GUT: Eindeutige Thread-IDs
chat("user_max_2024", "Frage 1")
chat("user_emma_2024", "Frage 1")

# NICHT GUT: Identische Thread-IDs für verschiedene User
chat("user", "Max Frage")
chat("user", "Emma Frage")  # ❌ Histories vermischt!
```

### 2. State verstehen
```python
# State ist IMMER eine Liste von Messages
state = graph.get_state(config)
messages = state.values["messages"]
# → [SystemMessage, HumanMessage, AIMessage, HumanMessage, AIMessage, ...]
```

### 3. Checkpointer-Strategy
```python
# v0.2: MemorySaver (Development)
checkpointer = MemorySaver()

# Später: SqliteSaver (Production)
from langgraph.checkpoint.sqlite import SqliteSaver
checkpointer = SqliteSaver(":memory:")
```

### 4. Error Handling
```python
try:
    response = graph.invoke(input_state, config=config)
except Exception as e:
    print(f"Error in thread {thread_id}: {e}")
    # Der State wird bei Error nicht corrupted
    # Nächster Aufruf wird fortgesetzt
```